In [31]:
### JSON PARSER
import json
import re
from typing import Optional

EMOTIONS_MAP = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

def parse_agent_output(raw: str) -> Optional[dict]:
    # 1
    result = _try_parse(raw.strip())
    if result and _validate_schema(result):
        return _normalize(result)
    # 2
    fenced = _extract_fenced_json(raw)
    if fenced:
        result = _try_parse(fenced)
        if result and _validate_schema(result):
            return _normalize(result)
    # 3
    braced = _extract_braced_json(raw)
    if braced:
        result = _try_parse(braced)
        if result and _validate_schema(result):
            return _normalize(result)
    # 4 
    return _regex_extract(raw)


def _try_parse(text: str) -> Optional[dict]:
    try:
        return json.loads(text)
    except (json.JSONDecodeError, ValueError):
        pass

    fixed = re.sub(r",\s*([}\]])", r"\1", text)
    try:
        return json.loads(fixed)
    except (json.JSONDecodeError, ValueError):
        pass

    fixed = text.replace("'", '"')
    try:
        return json.loads(fixed)
    except (json.JSONDecodeError, ValueError):
        pass

    return None

def _validate_schema(data: dict) -> bool:
    if not isinstance(data, dict):
        return False
    if "emotions" not in data:
        return False
    emotions = data["emotions"]
    if not isinstance(emotions, dict):
        return False
    return any(e in emotions for e in EMOTIONS_MAP)

def _normalize(data: dict) -> dict:
    emotions = {}
    confidence = {}
    for emo in EMOTIONS_MAP:
        raw_val = data.get("emotions", {}).get(emo, 0)
        if isinstance(raw_val, (int, float)):
            emotions[emo] = 1 if raw_val >= 0.5 else 0
        elif isinstance(raw_val, bool):
            emotions[emo] = 1 if raw_val else 0
        elif isinstance(raw_val, str):
            emotions[emo] = 1 if raw_val.lower() in ("1", "true", "yes") else 0
        else:
            emotions[emo] = 0
        
        raw_conf = data.get("confidence", {}).get(emo, 0.5)
        try:
            confidence[emo] = float(raw_conf)
        except (ValueError, TypeError):
            confidence[emo] = 0.5
        
    reasoning = data.get("reasoning", "")
    if not isinstance(reasoning, str):
        reasoning = str(reasoning)
    
    return {
        "emotions": emotions,
        "confidence": confidence,
        "reasoning": reasoning,
    }


def _extract_fenced_json(text: str) -> Optional[str]:
    pattern = r"```(?:json)?\s*\n?(.*?)\n?\s*```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None

def _extract_braced_json(text:str) -> Optional[str]:
    start = text.find("{")
    if start == -1:
        return None
    
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[start: i + 1]
    return None

def _regex_extract(text: str) -> Optional[dict]:
    emotions = {}
    confidence = {}

    for emo in EMOTIONS_MAP:
        pattern = rf'["\']?{emo}["\']?\s*:\s*([01])'
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            emotions[emo] = int(match.group(1))
        else:
            emotions[emo] = 0

        conf_pattern = rf'["\']?{emo}["\']?\s*:\s*(0\.\d+|1\.0|0|1)'
        conf_section = re.search(r'"confidence"\s*:\s*\{([^}]+)\}', text, re.DOTALL)
        if conf_section:
            conf_match = re.search(conf_pattern, conf_section.group(1))
            if conf_match:
                confidence[emo] = float(conf_match.group(1))
            else:
                confidence[emo] = 0.5
        else:
            confidence[emo] = 0.5

    if not any(v == 1 for v in emotions.values()):
        return None
    
    reasoning_match = re.search(r'"reasoning"\s*:\s*"([^"]*)"', text)
    reasoning = reasoning_match.group(1) if reasoning_match else ""

    return {
        "emotions": emotions,
        "confidence": confidence,
        "reasoning": reasoning,
    }


if __name__ == __name__:

    test1 = '{"emotions": {"joy": 1, "sadness": 0, "anger": 0}, "confidence": {"joy": 0.9, "sadness": 0.1, "anger": 0.2}, "reasoning": "The text is clearly positive."}'
    test2 = 'Here is the analysis:\n```json\n{"emotions": {"fear": 1, "surprise": 1}, "confidence": {"fear": 0.8, "surprise": 0.6}, "reasoning": "The situation is alarming."}\n```\nLet me know if you need anything else.'
    test3 = 'After careful analysis of the input, I concluded: {"emotions": {"disgust": 1, "anger": 1}, "confidence": {"disgust": 0.75, "anger": 0.85}, "reasoning": "Strong negative reaction detected."} Hope this helps!'
    test4 = 'The agent output was corrupted. Detected emotions: joy: 1, sadness: 0, "reasoning": "Partial output recovered by regex."'

    # 1
    print(parse_agent_output(test1))
    # 2
    print(parse_agent_output(test2))
    # 3
    print(parse_agent_output(test3))
    # 4
    print(parse_agent_output(test4))


{'emotions': {'anger': 0, 'disgust': 0, 'fear': 0, 'joy': 1, 'sadness': 0, 'surprise': 0}, 'confidence': {'anger': 0.2, 'disgust': 0.5, 'fear': 0.5, 'joy': 0.9, 'sadness': 0.1, 'surprise': 0.5}, 'reasoning': 'The text is clearly positive.'}
{'emotions': {'anger': 0, 'disgust': 0, 'fear': 1, 'joy': 0, 'sadness': 0, 'surprise': 1}, 'confidence': {'anger': 0.5, 'disgust': 0.5, 'fear': 0.8, 'joy': 0.5, 'sadness': 0.5, 'surprise': 0.6}, 'reasoning': 'The situation is alarming.'}
{'emotions': {'anger': 1, 'disgust': 1, 'fear': 0, 'joy': 0, 'sadness': 0, 'surprise': 0}, 'confidence': {'anger': 0.85, 'disgust': 0.75, 'fear': 0.5, 'joy': 0.5, 'sadness': 0.5, 'surprise': 0.5}, 'reasoning': 'Strong negative reaction detected.'}
{'emotions': {'anger': 0, 'disgust': 0, 'fear': 0, 'joy': 1, 'sadness': 0, 'surprise': 0}, 'confidence': {'anger': 0.5, 'disgust': 0.5, 'fear': 0.5, 'joy': 0.5, 'sadness': 0.5, 'surprise': 0.5}, 'reasoning': 'Partial output recovered by regex.'}


In [29]:
import re

text = '{"name": "Era", "age": 25,}'
re.sub(r",\s*([}\]])", r"\1", text)

text = """```json {"red": 1, "blue": 2,}\n```json {"blue": 2, "green": 3}```"""
pattern = r"```(?:json)?\s*\n?(.*?)\n?\s*```"
m = re.search(pattern, text,re.DOTALL)
str(m.group(1))

text = '{"reason": "Model thinks the sentiment is feeling lost and high energy", "anger": 1, "crazy": 0}'
emo = "crazy"
pattern = rf'["\']?{emo}["\']?\s*:\s*([01])'
re.search(pattern, text, re.IGNORECASE)

<re.Match object; span=(85, 95), match='"crazy": 0'>

In [ ]:
### COST TRACKER - token usage and latency

import time
from dataclasses import dataclass, field, FrozenInstanceError

@dataclass
class RequestStats:
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0
    latency_seconds: float = 0.0

    def __post_init__(self): # runs after init, post-init validation
        if self.completion_tokens < 0:
            raise ValueError("sr must be positive")

@dataclass
class CostTracker:
    requests: list[RequestStats] = field(default_factory=list)
    _start_time: float = 0.0

    def start_timer(self):
        self._start_time = time.time()

    def elapsed_seconds(self) -> float:
        return time.time() - self._start_time
    
    def record_request(
        self,
        prompt_tokens: int = 0,
        completion_tokens: int = 0,
        total_tokens: int = 0,
        latency_seconds: float = 0.0,
    ):
        self.requests.append(RequestStats(
            prompt_tokens=prompt_tokens,
            completion_tokens=completion_tokens,
            total_tokens=total_tokens,
            latency_seconds=latency_seconds,
        ))

    @property
    def total_prompt_tokens(self) -> int:
        return sum(r.prompt_tokens for r in self.requests)
    
    @property
    def total_completion_tokens(self) -> int:
        return sum(r.completion_tokens for r in self.requests)

    @property
    def total_tokens(self) -> int:
        return sum(r.total_tokens for r in self.requests)
    
    @property
    def total_latency(self) -> float:
        return sum(r.latency_seconds for r in self.requests)

    @property
    def num_requests(self) -> int:
        return len(self.requests)
    
    def summary(self) -> dict:
        n = self.num_requests
        return {
            "num_requests": n,
            "total_prompt_tokens": self.total_prompt_tokens,
            "total_completion_tokens": self.total_completion_tokens,
            "total_tokens": self.total_tokens,
            "total_latency_seconds": round(self.total_latency, 2),
            "wall_clock_seconds": round(self.elapsed_seconds(), 2),
            "avg_tokens_per_request": round(self.total_tokens / n, 1) if n else 0,
            "avg_latency_per_request": round(self.total_latency / n, 3) if n else 0,
        }

rs = RequestStats()
rs.prompt_tokens += 1
rs

@dataclass(frozen=True)
class Point:
    x: float
    y: float

p = Point(1.0, 2.0)
print(hash(p))      # works
try:
    p.x = 5.0 
except (FrozenInstanceError):
    print("cannot modify frozen dataclass")

-3550055125485641917
cannot modify frozen dataclass


In [ ]:
ct = CostTracker()
ct.start_timer()

5.5789947509765625e-05

In [49]:
ct.elapsed_seconds


12.1200430393219

In [52]:
### BASE AGENT

import time
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Optional

from openai import OpenAI


@dataclass
class AgentOutput:
    emotions: dict[str, int]
    confidence: dict[str, float]
    reasoning: str
    raw_response: str
    prompt_tokens: int = 0
    completion_tokens: int = 0
    parse_success: bool = True


class BaseAgent(ABC):
    """Abstract base class for all agents"""

    def __init__(
        self,
        name: str,
        endpoint: str = "http://localhost:8000/v1",
        api_key: str = "",
        model: str = "Qwen2.5-14B-Instruct",
        max_tokens: int = 512,
        temperature: float = 0.7,
        top_p: float = 0.95,
        seed: int = 42,
        cost_tracker: Optional[CostTracker] = None,
    ):
        self.name = name
        self.model = model
        self.max_tokens = max_tokens
        self.temperature = temperature
        self.top_p = top_p
        self.seed = seed
        self.cost_tracker = cost_tracker

        self.client = OpenAI(base_url=endpoint, api_key=api_key)

    @abstractmethod
    def get_system_prompt(self) -> str:
        """Return system prompt for this agent"""
        ...

    def predict(self, user_message: str) -> AgentOutput:
        system_prompt = self.get_system_prompt()

        start = time.time()
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message},
            ],
            max_tokens=self.max_tokens,
            temperature=self.temperature,
            top_p=self.top_p,
            seed=self.seed,
        )
        latency = time.time() - start

        raw = response.choices[0].message.content or ""
        usage = response.usage
        prompt_tokens = usage.prompt_tokens if usage else 0
        completion_tokens = usage.completion_tokens if usage else 0
        total_tokens = usage.total_tokens if usage else 0

        if self.cost_tracker:
            self.cost_tracker.record_request(
                prompt_tokens=prompt_tokens,
                completion_tokens=completion_tokens,
                total_tokens=total_tokens,
                latency_seconds=latency,
            )

        parsed = parse_agent_output(raw)
        if parsed is None:
            return AgentOutput(
                emotions={e: 0 for e in EMOTIONS_MAP},
                confidence={e: 0.5 for e in EMOTIONS_MAP},
                reasoning="[PARSE FAILURE]",
                raw_response=raw,
                prompt_tokens=prompt_tokens,
                completion_tokens=completion_tokens,
                parse_success=False,
            )

        return AgentOutput(
            emotions=parsed["emotions"],
            confidence=parsed["confidence"],
            reasoning=parsed["reasoning"],
            raw_response=raw,
            prompt_tokens=prompt_tokens,
            completion_tokens=completion_tokens,
            parse_success=True,
        )

In [55]:
class PromptAgent(BaseAgent):
    def __init__(self, system_prompt: str, **kwargs):
        super(PromptAgent, self).__init__(**kwargs)
        self._system_prompt = system_prompt
    
    def get_system_prompt(self) -> str:
        return self._system_prompt

pa = PromptAgent("")

TypeError: BaseAgent.__init__() missing 1 required positional argument: 'name'

In [50]:
!pip install openai

  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.1 MB/s  0:00:00
Using cached distro-1.9.

In [3]:
import time
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import TYPE_CHECKING, Optional

from vllm import SamplingParams

from src.utils.json_parser import parse_agent_output
from src.utils.cost_tracker import CostTracker

if TYPE_CHECKING:
    from vllm import LLM


@dataclass
class AgentOutput:
    emotions: dict[str, int]
    confidence: dict[str, float]
    reasoning: str
    raw_response: str
    prompt_tokens: int = 0
    completion_tokens: int = 0
    parse_success: bool = True


class BaseAgent(ABC):
    """Abstract base class for agents"""

    def __init__(
        self,
        name: str,
        llm: "LLM",
        model: str = "Qwen/Qwen2.5-14B-Instruct",
        max_tokens: int = 512,
        temperature: float = 0.7,
        top_p: float = 0.95,
        seed: int = 42,
        cost_tracker: Optional[CostTracker] = None,
    ):
        self.name = name
        self.model = model
        self.cost_tracker = cost_tracker

        self.llm = llm
        self.sampling_params = SamplingParams(
            temperature=temperature,
            top_p=top_p,
            max_tokens=max_tokens,
            seed=seed,
        )

    @abstractmethod
    def get_system_prompt(self) -> str:
        """Returning system prompt for the agent"""
        ...
    
    def predict(self, user_message: str, cot_block: str = "") -> AgentOutput:
        """Send message to LLM and parse the response"""
        system_prompt = self.get_system_prompt()
        if system_prompt and cot_block:
            system_prompt = system_prompt + '\n\n' + cot_block
        elif cot_block:
            system_prompt = cot_block
        
        start = time.time()
        outputs = self.llm.chat(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message},
            ],
            sampling_params=self.sampling_params,
        )
        latency = time.time() - start

        output = outputs[0]
        raw = output.outputs[0].text
        prompt_tokens = len(output.prompt_token_ids)
        completion_tokens = len(output.outputs[0].token_ids)
        total_tokens = prompt_tokens + completion_tokens

        if self.cost_tracker:
            self.cost_tracker.record_request(
                prompt_tokens=prompt_tokens,
                completion_tokens=completion_tokens,
                total_tokens=total_tokens,
                latency=latency,
            )

        parsed = parse_agent_output(raw)
        if parsed is None:
            # Return default all-zeros on parse failure
            from src.data.loader import EMOTIONS
            return AgentOutput(
                emotions={e: 0 for e in EMOTIONS},
                confidence={e: 0.5 for e in EMOTIONS},
                reasoning="[PARSE FAILURE]",
                raw_response=raw,
                prompt_tokens=prompt_tokens,
                completion_tokens=completion_tokens,
                parse_success=False,
            )

        return AgentOutput(
            emotions=parsed["emotions"],
            confidence=parsed["confidence"],
            reasoning=parsed["reasoning"],
            raw_response=raw,
            prompt_tokens=prompt_tokens,
            completion_tokens=completion_tokens,
            parse_success=True,
        )


In [4]:
class PromptAgent(BaseAgent):

    def __init__(self, system_prompt: str, **kwargs):
        super().__init__(**kwargs)
        self._system_prompt = system_prompt

    def get_system_prompt(self) -> str:
        return self._system_prompt


In [ ]:
import numpy as np
import torch
import yaml
from torch.utils.data import Dataset


class EmotionChatDataset(Dataset):
    def __init__(self, formatted_examples: list[dict]):
        self.examples = formatted_examples
    
    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        return {
            "input_ids": torch.tensor(ex["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(ex["attention_maks"], dtype=torch.long),
            "labels": torch.tensor(ex["labels"], dtype=torch.long),
            "weight": torch.tensor(ex["weight"], dtype=torch.float),
        }
    
def collate_fn(batch: list[dict]) -> dict:
    """Pad batch to max lenght"""
    max_len = max(len(ex["input_ids"]) for ex in batch)

    input_ids = torch.zeros(len(batch), max_len, dtype=torch.long)
    attention_mask = torch.zeros(len(batch), max_len, dtype=torch.long)
    labels = torch.full((len(batch), max_len), -100, dtype=torch.long)
    weights = torch.zeros(len(batch), dtype=torch.float)

    for i, ex in enumerate(batch):
        n = len(ex["input_ids"])
        input_ids[i, :n] = ex["input_ids"]
        attention_mask[i, :n] = ex["attention_mask"]
        labels[i, :n] = ex["labels"]
        weights[i] = ex["weight"]
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "weights": weights,
    }


def contrastive_loss(
    hidden_states: torch.Tensor,
    label_matrix: torch.Tensor,
    margin: float = 0.5,
    temperature: float = 0.1,
) -> torch.Tensor:
    from torch.nn.functional import normalize, cosine_similarity

    hidden_states = normalize(hidden_states, dim=-1)
    n_emotions = label_matrix.size(1)
    total_loss = torch.tensor(0.0, device=hidden_states.device)
    n_terms = 0

    for e in range(n_emotions):
        pos_mask = label_matrix[:, e] == 1
        neg_mask = label_matrix[:, e] == 0

        pos_idx = pos_mask.nonzero(as_tuple=True)[0]
        neg_idx = neg_mask.nonzero(as_tuple=True)[0]

        if len(pos_idx) == 0 or len(neg_idx) == 0:
            continue

        

False

In [ ]:
import json
from itertools import combinations
from pathlib import Path

from src.data.loader import EMOTIONS, LANGUAGES


def pairwise_disagreement_rate(agent_outputs: list[dict]) -> float:
    """Fraction of (agent_i, agent_j) pairs that disagree on >=1 emotion label"""
    if len(agent_outputs) < 2:
        return 0.0

    pairs = list(combinations(agent_outputs, 2))
    disagreements = 0
    for a, b in pairs:
        emotions_a = a.get("emotions", {})
        emotions_b = b.get("emotions", {})
        all_keys = set(emotions_a) | set(emotions_b)
        if any(emotions_a.get(k, 0) != emotions_b.get(k, 0) for k in all_keys):
            disagreements += 1

    return disagreements / len(pairs)


def per_emotion_disagreement(agent_outputs: list[dict]) -> dict[str, float]:
    """Per-emotion disagreement rate across all agent pairs"""
    if len(agent_outputs) < 2:
        return {emo: 0.0 for emo in EMOTIONS}

    pairs = list(combinations(agent_outputs, 2))
    counts = {emo: 0 for emo in EMOTIONS}
    for a, b in pairs:
        emotions_a = a.get("emotions", {})
        emotions_b = b.get("emotions", {})
        for emo in EMOTIONS:
            if emotions_a.get(emo, 0) != emotions_b.get(emo, 0):
                counts[emo] += 1

    return {emo: counts[emo] / len(pairs) for emo in EMOTIONS}


def compute_diversity_report(predictions_jsonl_path: str) -> dict:
    """Compute pairwise disagreement statistics from a run_config_b JSONL output"""
    records = []
    with open(predictions_jsonl_path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    if not records:
        return {"error": "No records found in JSONL file."}

    # Per-instance pairwise disagreement
    instance_disagree = []
    for rec in records:
        agent_outputs = rec.get("agent_predictions", [])
        instance_disagree.append(pairwise_disagreement_rate(agent_outputs))

    # Group records by language
    by_language: dict[str, list[dict]] = {}
    for rec in records:
        lang = rec.get("language", "unknown")
        by_language.setdefault(lang, []).append(rec)

    per_language: dict[str, float] = {}
    per_language_per_emotion: dict[str, dict[str, float]] = {}
    for lang, lang_records in by_language.items():
        lang_disagree = [
            pairwise_disagreement_rate(r.get("agent_predictions", []))
            for r in lang_records
        ]
        per_language[lang] = round(sum(lang_disagree) / len(lang_disagree), 4)

        # Per-emotion for this language
        emo_counts = {emo: 0.0 for emo in EMOTIONS}
        for rec in lang_records:
            emo_d = per_emotion_disagreement(rec.get("agent_predictions", []))
            for emo in EMOTIONS:
                emo_counts[emo] += emo_d[emo]
        per_language_per_emotion[lang] = {
            emo: round(emo_counts[emo] / len(lang_records), 4)
            for emo in EMOTIONS
        }

    # Global per-emotion: average across all instances
    global_emo_counts = {emo: 0.0 for emo in EMOTIONS}
    for rec in records:
        emo_d = per_emotion_disagreement(rec.get("agent_predictions", []))
        for emo in EMOTIONS:
            global_emo_counts[emo] += emo_d[emo]
    per_emotion_global = {
        emo: round(global_emo_counts[emo] / len(records), 4)
        for emo in EMOTIONS
    }

    return {
        "overall_disagreement_rate": round(sum(instance_disagree) / len(instance_disagree), 4),
        "per_language": per_language,
        "per_emotion": per_emotion_global,
        "per_language_per_emotion": per_language_per_emotion,
        "n_instances": len(records),
    }


def print_diversity_report(report: dict) -> None:
    """Pretty-print a diversity report to stdout."""
    if "error" in report:
        print(f"Error: {report['error']}")
        return

    print(f"Pairwise Disagreement Report  (n={report['n_instances']})")
    print("=" * 52)
    print(f"Overall disagreement rate: {report['overall_disagreement_rate']:.4f}")

    print("\nPer-language disagreement rate:")
    for lang in sorted(report["per_language"]):
        print(f"  {lang:6s}  {report['per_language'][lang]:.4f}")

    print("\nPer-emotion disagreement rate (global):")
    for emo in EMOTIONS:
        print(f"  {emo:10s}  {report['per_emotion'][emo]:.4f}")



report = compute_diversity_report(jsonl)
print_diversity_report(report)